# P127 — Jukebox: un modelo generativo de música

## 1. Título y paper

**Paper:** *Jukebox: A Generative Model for Music*  
**Autoría:** Prafulla Dhariwal, Heewoo Jun, Christine Payne, Jong Wook Kim, Alec Radford, Ilya Sutskever  
**Año y venue:** 2020 · arXiv:2005.00341  
**Nivel:** L3 · **Motor:** `jukebox`  
**Ficha completa:** [`P127_jukebox`](../../papers/foundational/P127_jukebox/README.md)

**Hito:** Genera canciones con voz cantada reconocible modelando códigos discretos en tres escalas temporales, en vez de la forma de onda directamente.

- [arXiv:2005.00341](https://arxiv.org/abs/2005.00341)

> Este notebook implementa una **miniatura** del mecanismo. No reproduce el experimento original ni sus métricas: reproduce la idea para que se pueda inspeccionar y discutir.


## 2. Objetivos

1. Explicar qué problema resolvió el paper: Cuatro minutos de audio a 44,1 kHz son más de diez millones de muestras. Ningún modelo autorregresivo opera sobre esa longitud, y comprimir a una sola escala obliga a elegir entre estructura larga y detalle tímbrico.
2. Ejecutar una implementación mínima de la propuesta: Un cuantizador vectorial jerárquico que codifica el audio en tres niveles de compresión distintos, y un modelo autorregresivo por nivel: el grueso decide la estructura y los finos reconstruyen el timbre condicionados por él.
3. Predecir el resultado antes de ejecutar, y contrastar la predicción con la salida.
4. Identificar al menos una limitación de la miniatura y una del paper original.
5. Conectar el hito con el siguiente eslabón de la ruta.


## 3. Prerrequisitos

- Python 3.11+ y el paquete del programa instalado (`pip install -e .`).
- Haber leído la guía [método de lectura en 5 pasadas](../../papers/guides/METODO_DE_LECTURA_EN_5_PASADAS.md).
- Hitos previos:
- P119
- P38


## 4. Intuición

Cuatro minutos de canción son diez millones y medio de muestras. Antes de modelar hay que comprimir — y a cuánto se comprime decide qué se puede aprender.


## 5. Concepto mínimo

```text
Jerarquía de códigos:
  128×  →  82 687 códigos   estructura, letra, forma
   32×  → 330 750 códigos   armonía, textura
    8×  → 1 323 000 códigos timbre, detalle

ventana fija → cuanto más fino el nivel, MENOS tiempo abarca
```


## 6. Código explicado

El motor aísla el mecanismo del paper con datos de juguete y salida inspeccionable.


In [ ]:
import json
import pathlib
import sys

ROOT = pathlib.Path.cwd()
while not (ROOT / "pyproject.toml").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))

from ai_evolution.papers_lab import run_paper_lab


def show(value):
    print(json.dumps(value, ensure_ascii=False, indent=2))


In [ ]:
r = run_paper_lab('jukebox', seed=7)['result']
show(r)

## 7. Predicción antes de ejecutar

1. ¿Cuántas muestras son cuatro minutos a 44,1 kHz?
2. ¿Cuánto tiempo abarca cada nivel con la misma ventana?
3. ¿Cubre alguno la canción entera?

> Escribe tu respuesta aquí antes de continuar.


## 8. Experimento controlado

Se varía una sola cosa y se observa el efecto.


In [ ]:
for semilla in (1, 7, 42):
    r = run_paper_lab('jukebox', seed=semilla)
    print(f'semilla {semilla:>2} · evidencia principal:')
    for e in r['evidence']:
        print('   +', e)
    break  # determinista: basta una para ver la estructura
for semilla in (1, 7, 42):
    r = run_paper_lab('jukebox', seed=semilla)['result']
    print(f'semilla {semilla:>2} → claves: {list(r)[:4]}')

## 9. Salida interpretable

**10 584 000** muestras. Con una ventana de 8 192 códigos, el nivel grueso abarca **23,8 s** y el fino **1,5 s**: 16× de diferencia. Y **ninguno cubre la canción** — 23,8 s de 240. Comprimir 128× cuesta fidelidad: el error de reconstrucción es **3,9×** peor que a 8×.


## 10. Comentario pedagógico

Ese «ninguno cubre la canción» es el límite real de Jukebox. Hay que generar por ventanas solapadas, y por eso el resultado suena a fragmentos bien hechos que no llegan a formar una pieza. La jerarquía compra tres horizontes, no un horizonte infinito.


## 11. Error o anti-patrón deliberado

Anti-patrón: elegir la compresión pensando solo en la fidelidad.


In [ ]:
print('Comprimir poco conserva el timbre y deja la ventana viendo dos segundos.')
print('Comprimir mucho abarca la forma y borra el detalle que hace que suene a musica.')
print('Por eso son tres niveles condicionados, no una eleccion.')

## 12. Corrección

La jerarquía y lo que cuesta:


In [ ]:
r = run_paper_lab('jukebox', seed=3)['result']
for n in r['jerarquia']:
    print(n)
print('fidelidad:', r['fidelidad_por_compresion'])
print('cobertura:', r['cobertura_temporal_por_nivel'])

## 13. Desafío guiado

Explica por qué una sola escala de compresión no puede servir para estructura y timbre a la vez, y qué habría que cambiar para que una sola bastara.


In [ ]:
r = run_paper_lab('jukebox', seed=3)['result']
show(r)

## 14. Desafío autónomo

Toma una pieza que conozcas y anota a qué escala temporal ocurre cada cosa que la hace reconocible: el timbre, el ritmo, la armonía, la forma. Compáralo con los tres niveles.


## 15. Evidencia de aprendizaje

Guarda tu tabla de escalas temporales y a qué nivel correspondería cada una.

Autoevaluación y respuestas esperadas: [ficha del paper](../../papers/foundational/P127_jukebox/README.md) · evaluación formal: [`assessments/papers/P127_jukebox.md`](../../assessments/papers/P127_jukebox.md)


## 16. Cierre

Abre la ruta de medios: generar deja de ser entender y pasa a plantear de quién es lo generado.


## 17. Conexión con el siguiente hito

- P129

Ruta completa: [`papers/ROADMAP.md`](../../papers/ROADMAP.md)
